# **Phase 1: Execution Path Understanding.**

In [5]:
import duckdb
import time
import psutil
import os
import time
import glob
import tempfile

**1A - Parser Presence**

In [7]:
import duckdb

# Intentionally misspelling SELECT and forgetting the GROUP BY clause to trigger a Syntax Error
query = """
SELCT AIRLINE, count(*) as delayed_flights 
FROM read_csv_auto('flights.csv') 
WHERE DEPARTURE_DELAY > 15;
"""

print("Testing the Postgres Parser...")

# This will fail immediately and throw a ParserException
result = duckdb.sql(query)
result.show()

Testing the Postgres Parser...


ParserException: Parser Error: syntax error at or near "SELCT"

LINE 2: SELCT AIRLINE, count(*) as delayed_flights 
        ^

**1B - Binder Presence**

In [8]:
import duckdb

# Intentionally using incorrect column names to trigger the Logical Planner's Binder
query = """
SELECT CARRIER, count(*) as delayed_flights, avg(DEP_DELAY) as avg_delay 
FROM read_csv_auto('flights.csv') 
WHERE DEP_DELAY > 15 
GROUP BY CARRIER 
ORDER BY delayed_flights DESC;
"""

print("Testing Logical Planner (Binder)...")

# This will fail and throw the BinderException
result = duckdb.sql(query)
result.show()

Testing Logical Planner (Binder)...


BinderException: Binder Error: Referenced column "DEP_DELAY" not found in FROM clause!
Candidate bindings: "DEPARTURE_DELAY", "ELAPSED_TIME", "AIRLINE_DELAY", "WEATHER_DELAY", "AIR_SYSTEM_DELAY"

**1C - Physical Planner & Optimizer (The DAG)**

In [9]:
plan = duckdb.sql(""" EXPLAIN SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay 
                              FROM read_csv_auto('flights.csv') 
                              WHERE DEPARTURE_DELAY > 15 
                              GROUP BY AIRLINE 
                              ORDER BY delayed_flights DESC """)
plan.show()

┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

**1D - Execution Engine**

In [10]:
query = """
SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay 
FROM read_csv_auto('flights.csv') 
WHERE DEPARTURE_DELAY > 15 
GROUP BY AIRLINE 
ORDER BY delayed_flights DESC;
"""

print("Running factory DuckDB (Vector Size = 2048)...")
start_time = time.time()

result = duckdb.sql(query)
result.show()

print(f"Finished in: {time.time() - start_time:.4f} seconds")

Running factory DuckDB (Vector Size = 2048)...
┌─────────┬─────────────────┬────────────────────┐
│ AIRLINE │ delayed_flights │     avg_delay      │
│ varchar │      int64      │       double       │
├─────────┼─────────────────┼────────────────────┤
│ WN      │          254138 │   52.4152271600469 │
│ AA      │          119456 │   64.3777122957407 │
│ DL      │          118136 │  62.63337170718494 │
│ UA      │          116153 │  64.74337296496861 │
│ EV      │           94024 │   68.5708010720667 │
│ OO      │           93665 │   66.4144344205413 │
│ B6      │           55779 │ 63.278294698721744 │
│ MQ      │           54572 │  64.13457450707322 │
│ NK      │           30641 │  66.44792924512907 │
│ US      │           28053 │  56.24917121163512 │
│ F9      │           20154 │  72.13853329363899 │
│ AS      │           17923 │  54.90498242481728 │
│ VX      │           10630 │ 59.373000940733775 │
│ HA      │            5234 │  48.65724111578143 │
└─────────┴─────────────────┴──────